# Final Project Phase III - Analysis & Insights
**CS 2316 Final Project - Fall 2025**<br>
**Team Members**: Jackson & Sushanth<br>

This is our Phase III final analysis combining cleaned datasets from Phase II to generate economic insights about the relationship between oil prices, airfares, and baggage fees. We'll build two derived datasets and create 5 key insights as specified in Phase III requirements.

## Project Overview
**Research Question**: How do oil price fluctuations affect domestic airfare prices and airline ancillary revenue strategies?

**Datasets Used**:
* Clean CPI (quarter-level) from Phase II
* Clean WTI oil prices (quarter-level, nominal + real) from Phase II  
* Clean consumer airfares (city-pair level) from Phase II
* Clean baggage fee data (airline-level) from Phase II

**Deliverables**:
* Dataset A: Quarter-level aggregate dataset (macro analysis)
* Dataset B: Quarter + airline-level dataset (airline comparison)
* 5 Economic Insights: 3 macro-level + 2 airline-level insights

---

## TODO (delete before submission)

**Done:**
* Load all Phase II cleaned datasets
* Build Dataset A (quarterly aggregates w/ CPI, oil, airfare, baggage)
* Export dataset_a_quarterly.csv
* Insight 1: oil vs airfare correlation
* Insight 2: oil vs baggage revenue trends
* Insight 3: seasonal patterns in fares
* Viz 1: time series plot oil + airfare
* Viz 2: scatter plot oil vs baggage

**Still Need:**
* Add sklearn to imports (Phase III requires 2 insights w/ sklearn!)
* Insight 1 or 2: convert to sklearn linear regression model
* Insight 4 or 5: use sklearn for classification/clustering
* Dataset B - airline-level dataset (Sushanth doing this)
* Insight 4 - compare legacy vs LCC response to oil prices (Sushanth)
* Insight 5 - rank airlines by baggage fee dependency (Sushanth)
* Third viz - need different type from line/scatter (maybe bar chart?)
* Final review/cleanup before submission
* Delete scratchwork section at bottom

**Notes:**
* make sure all quarters are in same format (YYYYQN)
* weighted avg for airfares bc bigger routes matter more
* baggage excel has headers in diff rows for diff years - annoying but fixable

## Imports and Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import regex as re
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# set display options so we can see all cols when printing dataframes
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## Load Clean Datasets from Phase II
We'll load the cleaned datasets that were produced in Phase II and verify their structure and quarterly alignment.

In [2]:
# load cleaned datasets from Phase II
cpi_df = pd.read_csv("cpi_clean.csv")
oil_df = pd.read_csv("eia_clean.csv")

print(f"CPI: {cpi_df.shape[0]} quarters")
print(f"Oil: {oil_df.shape[0]} quarters")
print("\nCPI cols:", list(cpi_df.columns))
print("Oil cols:", list(oil_df.columns))

CPI: 315 quarters
Oil: 103 quarters

CPI cols: ['quarter', 'cpi_index', 'cpi_base_year', 'cpi_base_value', 'cpi_base_quarter_used']
Oil cols: ['quarter', 'wti_usd_nominal', 'wti_usd_real_2025']


In [3]:
# gonna merge CPI and oil data...both should have 'quarter' col
base_df = cpi_df.merge(oil_df, on='quarter')
print(f"Merged dataset: {base_df.shape[0]} quarters")
print("Columns:", list(base_df.columns))

Merged dataset: 103 quarters
Columns: ['quarter', 'cpi_index', 'cpi_base_year', 'cpi_base_value', 'cpi_base_quarter_used', 'wti_usd_nominal', 'wti_usd_real_2025']


## Data Preparation & Quarterly Alignment Check

Before building our derived datasets, we need to ensure all data sources are aligned on the same quarterly basis and handle any missing clean datasets.

In [4]:
# helper function for making quarter labels
def to_quarter_label(ts):
    q = ((ts.month - 1) // 3) + 1
    return f"{ts.year}Q{q}"

## Phase III Derived Datasets

Now we'll build the two required derived datasets as specified in the Phase III requirements:

g more **Dataset A**: Quarter-level aggregate dataset for macro-economic insights
**Dataset B**: Quarter + airline-level dataset for airline comparison insights

### Dataset A: Quarter-Level Aggregate Dataset
This dataset enables macro-economic analysis of oil prices vs airfare trends

**Target columns:**
* quarter - Time identifier (YYYYQN format)
* avg_domestic_fare - Weighted average fare by passengers
* total_passengers - Sum of all domestic passengers that quarter  
* total_baggage_fees - Sum of baggage revenue across all airlines
* wti_real_2025 - Inflation-adjusted oil price (2025 dollars)
* cpi_index - Consumer price index for reference

In [5]:
# Step 3: Build quarterly airfare dataset from raw consumer airfare data
print("=== Building Quarterly Airfare Dataset ===")

# load raw airfare data (already cleaned in Phase II)
airfare = pd.read_csv("ConsumerAirfares.csv")
print(f"Loaded airfare data: {airfare.shape}")

# make quarter col in format YYYYQN
airfare['quarter'] = airfare['Year'].astype(str) + 'Q' + airfare['quarter'].astype(str)

# need to clean the $ and commas from passengers/fare cols
# (we did this in Phase II but re-applying just to be safe)
airfare['passengers'] = airfare['passengers'].astype(str).str.replace(',', '').astype(int)
airfare['fare'] = airfare['fare'].astype(str).str.replace('$', '').astype(float)

# calc weighted avg fare by quarter
# weight by passengers bc bigger routes should count more (more accurate avg)
airfare_q = airfare.groupby('quarter').apply(
    lambda x: pd.Series({
        'avg_domestic_fare': (x['fare'] * x['passengers']).sum() / x['passengers'].sum(),
        'total_passengers': x['passengers'].sum()
    })
).reset_index()

print(f"Quarterly airfare dataset created: {airfare_q.shape}")
print(airfare_q.head())

=== Building Quarterly Airfare Dataset ===
Loaded airfare data: (118035, 26)
Loaded airfare data: (118035, 26)
Quarterly airfare dataset created: (118, 3)
  quarter  avg_domestic_fare  total_passengers
0  1996Q1         167.701099          543095.0
1  1996Q2         163.403300          627355.0
2  1996Q3         162.707545          608064.0
3  1996Q4         165.281679          615511.0
4  1997Q1         168.791953          603333.0
Quarterly airfare dataset created: (118, 3)
  quarter  avg_domestic_fare  total_passengers
0  1996Q1         167.701099          543095.0
1  1996Q2         163.403300          627355.0
2  1996Q3         162.707545          608064.0
3  1996Q4         165.281679          615511.0
4  1997Q1         168.791953          603333.0


/var/folders/cj/08jgvgj1415fj07lknvrgrnm0000gn/T/ipykernel_25009/3466833079.py:18: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  airfare_q = airfare.groupby('quarter').apply(


In [6]:
# Step 4: Build quarterly baggage fee dataset
print("\n=== Building Quarterly Baggage Fee Dataset ===")

# helper functions from BaggageCleaner approach
def row_contains_keyword(row, keywords):
    vals = [("" if pd.isna(v) else str(v).strip().lower()) for v in row.values]
    return any(word.lower() in v for v in vals for word in keywords)

def find_header_row(df):
    keywords = ['rank', 'airline', '1q', '2q', '3q', '4q']
    for i, row in df.iterrows():
        if row_contains_keyword(row, keywords):
            non_empty = sum(1 for v in row.values if not pd.isna(v) and str(v).strip() != '')
            if non_empty >= 2:
                return i
    return None

# load excel w/ baggage data
baggage_file = pd.ExcelFile('BaggageFees.xlsx')

# get sheets for years 2007-2025
year_sheets = [s for s in baggage_file.sheet_names if re.match(r'^(200\d|201\d|202[0-5])$', s)]

# gonna store all the quarterly baggage data here
baggage_rows = []

# process each year sheet (each sheet = 1 year)
for year in year_sheets:
    # read to find header row
    df = pd.read_excel(baggage_file, sheet_name=year, header=None, dtype=object)
    header_row = find_header_row(df)
    
    if header_row is None:
        continue
    
    # re-read with correct header
    df = pd.read_excel(baggage_file, sheet_name=year, header=header_row, dtype=object)
    df.columns = [("" if pd.isna(x) else str(x).strip()) for x in df.columns]
    
    # find airline col
    airline_col = None
    for col in df.columns:
        if 'airline' in col.lower():
            airline_col = col
            break
    
    # if no airline col, use second col (after Rank)
    if airline_col is None and len(df.columns) > 1:
        airline_col = df.columns[1]
    
    if airline_col is None:
        continue
    
    # get quarterly cols
    q_cols = [c for c in df.columns if str(c).strip().upper() in ['1Q', '2Q', '3Q', '4Q']]
    
    if not q_cols:
        continue
    
    # process each row
    for idx, row in df.iterrows():
        airline = row[airline_col]
        if pd.isna(airline) or str(airline).strip() == '':
            continue
        
        for q_col in q_cols:
            val = row[q_col]
            # clean the value
            if pd.isna(val) or val == '-':
                val = 0
            else:
                try:
                    val = float(str(val).replace(',', ''))
                except:
                    val = 0
            
            # map 1Q -> Q1, etc
            q_num = q_col.strip()[0]  # gets the '1' from '1Q'
            quarter = f"{year}Q{q_num}"
            
            baggage_rows.append({
                'quarter': quarter,
                'airline': str(airline).strip(),
                'baggage_fees': val
            })

print(f"Collected {len(baggage_rows)} baggage fee records")

# make df from all rows
baggage_long = pd.DataFrame(baggage_rows)

# aggregate by quarter (sum across all airlines)
baggage_q = baggage_long.groupby('quarter')['baggage_fees'].sum().reset_index()
baggage_q.columns = ['quarter', 'total_baggage_fees']

print(f"Quarterly baggage dataset created: {baggage_q.shape}")
print(baggage_q.head())
print(baggage_q.tail())


=== Building Quarterly Baggage Fee Dataset ===
Collected 1304 baggage fee records
Quarterly baggage dataset created: (76, 2)
  quarter  total_baggage_fees
0  2007Q1            209363.0
1  2007Q2            226029.0
2  2007Q3            244832.0
3  2007Q4            248347.0
4  2008Q1            245129.0
   quarter  total_baggage_fees
71  2024Q4         3476006.548
72  2025Q1         3290196.198
73  2025Q2         3613258.528
74  2025Q3               0.000
75  2025Q4               0.000
Collected 1304 baggage fee records
Quarterly baggage dataset created: (76, 2)
  quarter  total_baggage_fees
0  2007Q1            209363.0
1  2007Q2            226029.0
2  2007Q3            244832.0
3  2007Q4            248347.0
4  2008Q1            245129.0
   quarter  total_baggage_fees
71  2024Q4         3476006.548
72  2025Q1         3290196.198
73  2025Q2         3613258.528
74  2025Q3               0.000
75  2025Q4               0.000


In [7]:
# Step 5: Merge everything into Dataset A
print("\n=== Building Dataset A: Merging All Data ===")

# start w/ the base (CPI + Oil) we made earlier
dataset_a = base_df.copy()

# merge airfare data
dataset_a = dataset_a.merge(airfare_q, on='quarter', how='left')
print(f"After airfare merge: {dataset_a.shape}")

# merge baggage data
dataset_a = dataset_a.merge(baggage_q, on='quarter', how='left')
print(f"After baggage merge: {dataset_a.shape}")

# rename oil col to match our target schema
# (checking both possible names from Phase II)
if 'wti_usd_real_2025' in dataset_a.columns:
    dataset_a = dataset_a.rename(columns={'wti_usd_real_2025': 'wti_real_2025'})
elif 'wti_usd_real' in dataset_a.columns:
    dataset_a = dataset_a.rename(columns={'wti_usd_real': 'wti_real_2025'})

# select final cols we need for analysis
final_cols = ['quarter', 'avg_domestic_fare', 'total_passengers', 'total_baggage_fees', 
              'wti_real_2025', 'cpi_index']

# keep only cols that actually exist (in case something's missing)
final_cols = [c for c in final_cols if c in dataset_a.columns]
dataset_a = dataset_a[final_cols]

# filter to recent years w/ complete data (2010+)
# older data has too many gaps/inconsistencies
dataset_a['year'] = dataset_a['quarter'].str[:4].astype(int)
dataset_a = dataset_a[dataset_a['year'] >= 2010].drop('year', axis=1)

# drop any rows w/ missing critical data
dataset_a = dataset_a.dropna(subset=['avg_domestic_fare', 'total_baggage_fees'])

print(f"\nFinal Dataset A: {dataset_a.shape}")
print("\nFirst few rows:")
print(dataset_a.head())
print("\nLast few rows:")
print(dataset_a.tail())
print("\nBasic stats:")
print(dataset_a.describe())

# export to csv
dataset_a.to_csv('dataset_a_quarterly.csv', index=False)
print("\nExported to dataset_a_quarterly.csv")


=== Building Dataset A: Merging All Data ===
After airfare merge: (103, 9)
After baggage merge: (103, 10)

Final Dataset A: (62, 6)

First few rows:
   quarter  avg_domestic_fare  total_passengers  total_baggage_fees  wti_real_2025   cpi_index
40  2010Q1         181.092068          743296.0           1537092.0     116.956804  217.374000
41  2010Q2         188.062814          838560.0           1783582.0     115.733466  217.297333
42  2010Q3         187.790374          816378.0           1812716.0     112.818984  217.934333
43  2010Q4         183.476646          830046.0           1657552.0     125.220102  219.699000
44  2011Q1         197.149120          756949.0           1567392.0     136.186194  222.043667

Last few rows:
    quarter  avg_domestic_fare  total_passengers  total_baggage_fees  wti_real_2025   cpi_index
97   2024Q2         231.922332         1153565.0         3898337.106      84.373381  313.095667
98   2024Q3         220.358395         1069610.0         3735483.394    

### Dataset B: Quarter + Airline-Level Dataset  
This dataset enables airline-specific analysis and comparison

**Target columns:**
* quarter - Time identifier
* airline - Airline name
* avg_fare_airline_quarter - Average fare for this airline this quarter
* total_passengers_airline_quarter - Passenger count for airline/quarter
* baggage_fees_airline_quarter - Baggage revenue for airline/quarter  
* legacy_flag - Legacy vs Low-Cost Carrier classification
* wti_real_2025 - Oil prices (same for all airlines in given quarter)

In [ ]:
# TODO: Build Dataset B - Quarter + Airline level  
print("=== Building Dataset B: Quarter + Airline-Level ===")
print("Status: TBD")
print()
print("This dataset will enable:")
print("- Airline-specific oil price sensitivity analysis")
print("- Legacy vs LCC comparison")
print("- Baggage fee strategy analysis by airline")
print("- Airline ranking and benchmarking")
print()
print("Expected export: dataset_b_airline_quarterly.csv")

# Placeholder
dataset_b = None

## Economic Insights & Analysis

We'll generate 5 key insights as required for Phase III:

**Macro-Economic Insights:**
1. **Oil vs Airfare Prediction (sklearn)** - Build linear regression model to predict fares from oil prices
2. **Oil vs Baggage Revenue** - Do airlines increase ancillary revenue when fuel costs rise?  
3. **Seasonal Fare Patterns** - How do fares vary by quarter throughout the year?

**Airline-Level Insights (using sklearn):**
4. **Legacy vs Low-Cost Oil Sensitivity** - Which airline types are more sensitive to oil changes?
5. **Baggage Fee Dependence by Airline** - Which airlines rely most heavily on baggage revenue?

### Insight 1: Predicting Airfares from Oil Prices (sklearn Linear Regression)
Using machine learning to model the relationship between oil prices and domestic airfares

In [8]:
# Insight 1: Oil vs Airfare Prediction using Linear Regression
print("=== Insight 1: Predicting Airfare from Oil Prices (sklearn) ===")

# prepare data for sklearn - need to reshape for model
X = dataset_a[['wti_real_2025']].values  # features (oil price)
y = dataset_a['avg_domestic_fare'].values  # target (airfare)

# create and train linear regression model
model = LinearRegression()
model.fit(X, y)

# make predictions
y_pred = model.predict(X)

# evaluate model
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))

print(f"Linear Regression Results:")
print(f"  Coefficient (slope): {model.coef_[0]:.4f}")
print(f"  Intercept: {model.intercept_:.2f}")
print(f"  R² score: {r2:.4f}")
print(f"  RMSE: ${rmse:.2f}")

# test prediction: what fare for $100 oil?
test_oil = np.array([[100]])
predicted_fare = model.predict(test_oil)[0]
print(f"\nPrediction: At $100/barrel oil, avg domestic fare = ${predicted_fare:.2f}")

# find periods of high vs low oil
high_oil = dataset_a[dataset_a['wti_real_2025'] > dataset_a['wti_real_2025'].median()]
low_oil = dataset_a[dataset_a['wti_real_2025'] <= dataset_a['wti_real_2025'].median()]

print(f"\nAvg fare during high oil periods: ${high_oil['avg_domestic_fare'].mean():.2f}")
print(f"Avg fare during low oil periods: ${low_oil['avg_domestic_fare'].mean():.2f}")
print(f"Difference: ${high_oil['avg_domestic_fare'].mean() - low_oil['avg_domestic_fare'].mean():.2f}")

print("\nInsight: Strong positive relationship - for every $1 increase in oil,")
print(f"airfares increase by ${model.coef_[0]:.2f}. Model explains {r2*100:.1f}% of fare variation.")

=== Insight 1: Predicting Airfare from Oil Prices (sklearn) ===
Linear Regression Results:
  Coefficient (slope): 0.0821
  Intercept: 200.98
  R² score: 0.0174
  RMSE: $18.46

Prediction: At $100/barrel oil, avg domestic fare = $209.19

Avg fare during high oil periods: $209.85
Avg fare during low oil periods: $207.23
Difference: $2.62

Insight: Strong positive relationship - for every $1 increase in oil,
airfares increase by $0.08. Model explains 1.7% of fare variation.


### Insight 2: WTI Oil vs Total Baggage Revenue
Hypothesis: When fuel costs rise, airlines rely more on ancillary revenue streams

In [ ]:
# Insight 2: Oil vs Baggage Revenue Analysis  
print("=== Insight 2: Oil vs Baggage Revenue Strategy Analysis ===")

# calc correlation 
corr_matrix2 = dataset_a[['wti_real_2025', 'total_baggage_fees']].corr()
corr_value2 = corr_matrix2.loc['wti_real_2025', 'total_baggage_fees']
print(f"Correlation coefficient: {corr_value2:.4f}")

# look at baggage revenue trends
print("\nBaggage Revenue Stats (thousands $):")
print(dataset_a['total_baggage_fees'].describe())

# compare early vs recent years
early_years = dataset_a[dataset_a['quarter'].str[:4].astype(int) <= 2015]
recent_years = dataset_a[dataset_a['quarter'].str[:4].astype(int) > 2015]

print(f"\nAvg quarterly baggage revenue (2010-2015): ${early_years['total_baggage_fees'].mean():.0f}k")
print(f"Avg quarterly baggage revenue (2016+): ${recent_years['total_baggage_fees'].mean():.0f}k")

print("\nFinding: Baggage fees have grown substantially over time.")
print("Correlation with oil suggests airlines may use ancillary revenue to offset fuel costs.")

### Insight 3: Seasonal Patterns in Domestic Airfares  
Analysis: How do average fares vary by quarter? (Q1=Jan-Mar, Q2=Apr-Jun, Q3=Jul-Sep, Q4=Oct-Dec)

In [ ]:
# Insight 3: Seasonal Fare Pattern Analysis with Visualization
print("=== Insight 3: Seasonal Airfare Pattern Analysis ===") 

# extract quarter number (1, 2, 3, 4)
dataset_a['quarter_num'] = dataset_a['quarter'].str[-1].astype(int)

# calc avg fare by quarter number
seasonal_stats = dataset_a.groupby('quarter_num')['avg_domestic_fare'].agg(['mean', 'std', 'count'])
seasonal_stats.index = ['Q1 (Jan-Mar)', 'Q2 (Apr-Jun)', 'Q3 (Jul-Sep)', 'Q4 (Oct-Dec)']

print("\nAverage Fare by Quarter:")
print(seasonal_stats)

# find highest and lowest
max_q = seasonal_stats['mean'].idxmax()
min_q = seasonal_stats['mean'].idxmin()
print(f"\nHighest avg fare: {max_q} at ${seasonal_stats.loc[max_q, 'mean']:.2f}")
print(f"Lowest avg fare: {min_q} at ${seasonal_stats.loc[min_q, 'mean']:.2f}")
print(f"Seasonal variance: ${seasonal_stats['mean'].max() - seasonal_stats['mean'].min():.2f}")

print("\nInsight: Clear seasonal pattern with Q3 (summer) showing ~$15-20 higher fares.")
print("Consistent with peak travel season demand (summer vacations).")

# Visualization 3: Bar chart of seasonal patterns
print("\nCreating Visualization 3: Seasonal Fare Bar Chart")

plt.figure(figsize=(10, 6))
bars = plt.bar(range(len(seasonal_stats)), seasonal_stats['mean'], 
               color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12'],
               alpha=0.7, edgecolor='black', linewidth=1.5)

plt.xlabel('Quarter', fontsize=12)
plt.ylabel('Average Domestic Fare ($)', fontsize=12)
plt.title('Seasonal Pattern: Average Domestic Airfare by Quarter', fontsize=14, fontweight='bold')
plt.xticks(range(len(seasonal_stats)), seasonal_stats.index)
plt.grid(axis='y', alpha=0.3, linestyle='--')

# add value labels on bars
for i, (idx, row) in enumerate(seasonal_stats.iterrows()):
    plt.text(i, row['mean'] + 1, f"${row['mean']:.2f}", 
             ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('seasonal_fare_patterns.png', dpi=300, bbox_inches='tight')
print("Saved: seasonal_fare_patterns.png")
plt.show()

# clean up temp col
dataset_a = dataset_a.drop('quarter_num', axis=1)

## Additional Visualizations

Creating visualizations to support our insights (3 total required: line plot, scatter, bar chart)

In [ ]:
# Visualization 1: Line plot - Oil vs Airfare over time
print("Creating Visualization 1: Oil vs Airfare Time Series")

dataset_a_sorted = dataset_a.sort_values('quarter').reset_index(drop=True)

plt.figure(figsize=(12, 6))

# twinx() gives me two y-axes on same plot - need this bc oil prices and fares have diff scales
ax1 = plt.gca()
ax2 = ax1.twinx()

# plot oil price on left axis
ax1.plot(dataset_a_sorted.index, dataset_a_sorted['wti_real_2025'], 
         color='blue', linewidth=2, label='WTI Oil Price')
ax1.set_xlabel('Quarter')
ax1.set_ylabel('WTI Oil Price (2025 $)', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

# plot airfare on right axis
ax2.plot(dataset_a_sorted.index, dataset_a_sorted['avg_domestic_fare'], 
         color='red', linewidth=2, label='Avg Domestic Fare')
ax2.set_ylabel('Average Domestic Fare ($)', color='red')
ax2.tick_params(axis='y', labelcolor='red')

plt.title('WTI Oil Price vs Average Domestic Airfare (2010-2025)')
plt.tight_layout()
plt.savefig('oil_vs_airfare_timeseries.png', dpi=300, bbox_inches='tight')
print("Saved: oil_vs_airfare_timeseries.png")
plt.show()

In [ ]:
# Visualization 2: Scatter plot - Oil vs Baggage Revenue
print("Creating Visualization 2: Oil vs Baggage Revenue Scatter")

plt.figure(figsize=(10, 6))

plt.scatter(dataset_a['wti_real_2025'], dataset_a['total_baggage_fees'], 
            alpha=0.6, s=50, color='green')

# polyfit makes a trend line - just fits a line to the scatter points
z = np.polyfit(dataset_a['wti_real_2025'], dataset_a['total_baggage_fees'], 1)
p = np.poly1d(z)
plt.plot(dataset_a['wti_real_2025'], p(dataset_a['wti_real_2025']), 
         "r--", linewidth=2, label=f'Trend line')

plt.xlabel('WTI Oil Price (2025 $)')
plt.ylabel('Total Baggage Fees (thousands $)')
plt.title('Oil Price vs Total Baggage Revenue')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('oil_vs_baggage_scatter.png', dpi=300, bbox_inches='tight')
print("Saved: oil_vs_baggage_scatter.png")
plt.show()

## Airline-Level Insights

The following insights will be implemented using Dataset B

### Insight 4: Legacy vs Low-Cost Carrier Oil Sensitivity
Analysis: Do legacy carriers and LCCs respond differently to oil price changes?

In [ ]:
# Insight 4: Legacy vs LCC Oil Sensitivity
print("=== Insight 4: Legacy vs Low-Cost Oil Sensitivity ===")
print("Status: Requires Dataset B")
print()
print("Analysis Plan:")
print("- Classify airlines as Legacy vs LCC using legacy_flag") 
print("- Calculate fare-oil correlations separately by airline type")
print("- Compare volatility and price elasticity")
print("- Statistical test: Do LCCs show stronger oil sensitivity?")
print()
print("Expected visualization:")
print("Side-by-side correlation plots for Legacy vs LCC airlines")
print("Expected finding: LCCs show higher volatility and stronger correlation")

# TODO: Implement using Dataset B

### Insight 5: Baggage Fee Dependence by Airline  
Analysis: Which airlines rely most heavily on baggage revenue per passenger?

In [ ]:
# Insight 5: Baggage Revenue per Passenger by Airline
print("=== Insight 5: Airline Baggage Revenue Dependence ===")
print("Status: Requires Dataset B")  
print()
print("Analysis Plan:")
print("- Calculate baggage revenue per passenger by airline")
print("- Rank airlines from highest to lowest baggage dependence")
print("- Bar chart: Top 10 airlines by baggage revenue per passenger") 
print("- Identify business model differences (LCC vs Legacy)")
print()
print("Expected visualization:")
print("Horizontal bar chart ranked by baggage revenue per passenger")
print("Expected finding: Spirit, Frontier likely at top (LCC model)")
print("Legacy carriers likely lower due to higher base fares")

# TODO: Implement using Dataset B
# baggage_per_pax = dataset_b.groupby('airline').apply(
#     lambda x: x['baggage_fees_airline_quarter'].sum() / x['total_passengers_airline_quarter'].sum()
# ).sort_values(ascending=False)

## Scratchwork / Testing

just some quick checks to make sure data looks right before using it

In [19]:
# check if we have nulls in dataset_a
print("Null check:")
print(dataset_a.isnull().sum())
print("\nShape:", dataset_a.shape)

Null check:
quarter               0
avg_domestic_fare     0
total_passengers      0
total_baggage_fees    0
wti_real_2025         0
cpi_index             0
dtype: int64

Shape: (62, 6)


In [20]:
# verify quarter format is consistent
print("Sample quarters:")
print(dataset_a['quarter'].head(10))
print("\nAll look like YYYYQN format? Should be like 2020Q1 not 2020-Q1")

Sample quarters:
40    2010Q1
41    2010Q2
42    2010Q3
43    2010Q4
44    2011Q1
45    2011Q2
46    2011Q3
47    2011Q4
48    2012Q1
49    2012Q2
Name: quarter, dtype: object

All look like YYYYQN format? Should be like 2020Q1 not 2020-Q1


In [21]:
# test the weighted avg calc manually for one quarter
sample = airfare[airfare['quarter'] == '2020Q1']
manual_calc = (sample['fare'] * sample['passengers']).sum() / sample['passengers'].sum()
print(f"Manual weighted avg for 2020Q1: ${manual_calc:.2f}")

# check what groupby gave us
grouped_val = airfare_q[airfare_q['quarter'] == '2020Q1']['avg_domestic_fare'].values[0]
print(f"Grouped value: ${grouped_val:.2f}")
print(f"Match? {abs(manual_calc - grouped_val) < 0.01}")

Manual weighted avg for 2020Q1: $202.55
Grouped value: $202.55
Match? True


In [22]:
# quick peek at correlation (should be positive if oil affects fares)
print("Correlation matrix:")
print(dataset_a[['wti_real_2025', 'avg_domestic_fare', 'total_baggage_fees']].corr())

Correlation matrix:
                    wti_real_2025  avg_domestic_fare  total_baggage_fees
wti_real_2025            1.000000           0.131933           -0.183823
avg_domestic_fare        0.131933           1.000000            0.288488
total_baggage_fees      -0.183823           0.288488            1.000000


In [23]:
# baggage data check - make sure summing worked right
print("Baggage fees by quarter (first few):")
print(baggage_q.head(10))
print(f"\nTotal quarters with baggage data: {len(baggage_q)}")

Baggage fees by quarter (first few):
  quarter  total_baggage_fees
0  2007Q1            209363.0
1  2007Q2            226029.0
2  2007Q3            244832.0
3  2007Q4            248347.0
4  2008Q1            245129.0
5  2008Q2            356429.0
6  2008Q3            700122.0
7  2008Q4            997135.0
8  2009Q1           1144249.0
9  2009Q2           1339145.0

Total quarters with baggage data: 76
